# P3 · Notebook 03b: Test the app's backend on a T4, then deploy it (GPU, ~15–20 min)

**What this is:** the app's server (`space/app.py` in the repo) is the code that runs on the Hugging Face Space: one address serves the phone app *and* the API (`/api/health`, `/api/generate`). This notebook runs **that exact server** on Colab's T4 and tests it the way the phone will use it, **before** we pay for a T4 Space.

1. Gets the code from GitHub and the cascade settings from Drive (`cascade_config_lowmem.json` from notebook 03a).
2. Starts the server and waits until the model is loaded and warmed up.
3. **Contract test:** every response must match `frontend/docs/API_CONTRACT.md` (the format your friend's UI expects).
4. **Error test:** empty prompt, too long, wrong mode, blocked prompt, broken request.
5. **Speed test:** each mode on held-out prompts, compared with notebook 03a's measurements.
6. **Several judges at once:** simultaneous requests (queueing, no crashes).
7. Saves the exact library versions that passed, and (only if you set `DEPLOY = True`) uploads everything to your Hugging Face Space.

**Before running:** notebooks 02b and 03a have finished. Runtime → **T4 GPU**.

## 1 · Settings

In [ ]:
import os
GITHUB_REPO = "Puneetha-S-Shankar/text-image-CV"
BRANCH = "main"
N_BENCH = int(os.environ.get("P3_N_BENCH", 20))   # held-out prompts per mode in the speed test
BURST = int(os.environ.get("P3_BURST", 6))         # simultaneous requests in the "several judges" test
DEPLOY = False        # True = upload to your Hugging Face Space at the end (needs HF_TOKEN in Colab Secrets)
HF_SPACE = ""         # e.g. "your-hf-username/p3-studio" (it will be PUBLIC: judges need to open it)
SAVE_TO_DRIVE = True

## 2 · Get the code
Public repository: nothing to do. **Private repository:** add a GitHub token with read access in Colab's 🔑 **Secrets** panel as `GITHUB_TOKEN` and switch on notebook access.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path(os.environ.get("P3_REPO_DIR", "/content/text-image-CV"))
if os.environ.get("P3_REPO_DIR"):
    print("Using the local checkout:", REPO_DIR)                    # offline testing
elif (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    public_url = f"https://github.com/{GITHUB_REPO}.git"
    if subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, public_url, str(REPO_DIR)]).returncode != 0:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, f"https://{token}@github.com/{GITHUB_REPO}.git",
                        str(REPO_DIR)], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", public_url], check=True)  # don't keep the token
print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h %s (%cr)"], capture_output=True, text=True).stdout)
SPACE = REPO_DIR / "space"
assert (SPACE / "app.py").exists(), f"{SPACE / 'app.py'} not found: is the repo up to date?"

## 3 · Install the server's libraries (~2 min)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "-r", str(SPACE / "requirements.txt"), "httpx",
                "huggingface_hub"], check=True)
print("Libraries installed.")

## 4 · Settings from Drive (the router, the gate, the measured modes)

In [ ]:
import json, shutil, time, base64, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

SHARED = Path(os.environ.get("P3_SHARED", "/content/P3_shared"))
IN_02B = Path(os.environ.get("P3_02B_OUT", "/content/P3_02b_out"))
OUT = Path(os.environ.get("P3_OUT", "/content/P3_03b_out"))
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SHARED = Path("/content/drive/MyDrive/P3_T2I/shared")
        IN_02B = Path("/content/drive/MyDrive/P3_T2I/02b_router_cascade")
        OUT = Path("/content/drive/MyDrive/P3_T2I/03b_backend")
    except Exception as e:
        print("Google Drive not available, using local folders:", e)
OUT.mkdir(parents=True, exist_ok=True)

CONFIG_FILE = next((SHARED / n for n in ("cascade_config_lowmem.json", "cascade_config.json") if (SHARED / n).exists()), None)
assert CONFIG_FILE, f"No cascade config in {SHARED}: run notebook 02b (and 03a) first."
for n in ("cascade_config_lowmem.json", "cascade_config.json"):
    (SPACE / n).unlink(missing_ok=True)
shutil.copy(CONFIG_FILE, SPACE / CONFIG_FILE.name)
CONFIG = json.load(open(CONFIG_FILE, encoding="utf-8"))
SETUP = (CONFIG.get("generator") or {}).get("setup", "fp16")
print(f"Config: {CONFIG_FILE.name} | generator setup: {SETUP} | levels: " + ", ".join(f"{r['name']} {r['res']}px" for r in CONFIG["rungs"]))
if CONFIG_FILE.name != "cascade_config_lowmem.json":
    print("NOTE: no low-memory config yet (notebook 03a). The server will run the float16 model.")

prompt_files = sorted(IN_02B.glob("prompts_*.json"))
TEST = [p for p in json.load(open(prompt_files[-1], encoding="utf-8")) if p["split"] == "test"] if prompt_files else []
if TEST:
    step = max(1, len(TEST) // N_BENCH)
    BENCH = [p["prompt"] for p in TEST[::step][:N_BENCH]]
else:
    BENCH = ["a photo of two dogs", "a photo of a red car and a blue bus", "a photo of a cat left of a dog",
             "a man riding a horse on a beach", "a bowl of fruit on a wooden table"] * (N_BENCH // 5 + 1)
    BENCH = BENCH[:N_BENCH]
print(f"Speed test: {len(BENCH)} held-out prompts per mode")

## 5 · Start the server (first time: downloads the model, ~3–6 min)
Exactly what the Space runs: `uvicorn app:app`. Its log goes to `server.log` in the results folder.

In [ ]:
import httpx

PORT = int(os.environ.get("P3_PORT", 7860))
BASE = f"http://127.0.0.1:{PORT}"
LOG = OUT / "server.log"
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", str(PORT)],
                          cwd=SPACE, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env={**os.environ})

def log_tail(n=40):
    return "\n".join(LOG.read_text(errors="replace").splitlines()[-n:])

t0, last_print, health = time.time(), 0, None
try:
    while True:
        try:
            health = httpx.get(f"{BASE}/api/health", timeout=5).json()
            if health["ready"]:
                break
            if health.get("error"):
                raise RuntimeError("The server couldn't load the model: " + health["error"])
        except httpx.TransportError:
            pass
        if server.poll() is not None:
            raise RuntimeError("The server stopped.")
        if time.time() - t0 > 1500:
            raise TimeoutError("Still not ready after 25 min.")
        if time.time() - last_print > 30:
            print(f"  loading... {time.time() - t0:.0f} s")
            last_print = time.time()
        time.sleep(3)
except Exception:
    print("---- last lines of server.log ----\n" + log_tail())
    raise
STARTUP_S = time.time() - t0
print(f"Ready after {STARTUP_S:.0f} s (includes download + warm-up):", health)

## 6 · Contract test: does every response match what the phone app expects?

In [ ]:
KINDS = {"object", "count", "colour", "position"}

def contract_problems(r: dict) -> list[str]:
    """Checks a /api/generate response against frontend/docs/API_CONTRACT.md."""
    p = []
    types_ = {"id": str, "image": str, "width": int, "height": int, "prompt": str, "mode": str, "passed": bool,
              "level_used": int, "levels_total": int, "attempts": list, "timing": dict, "checks": dict,
              "router": dict, "polish": dict, "model": str, "gpu": str}
    for k, t in types_.items():
        if not isinstance(r.get(k), t) or (t is int and isinstance(r.get(k), bool)):
            p.append(f"{k}: expected {t.__name__}, got {type(r.get(k)).__name__}")
    if p:
        return p
    if not r["image"].startswith("data:image/webp;base64,"):
        p.append("image is not a WebP data URL")
    if r["mode"] not in ("fast", "balanced", "quality"):
        p.append("bad mode echo")
    if not 0 <= r["level_used"] < r["levels_total"]:
        p.append("level_used out of range")
    for k in ("total_s", "generate_s", "check_s", "polish_s", "encode_s"):
        if not isinstance(r["timing"].get(k), (int, float)):
            p.append(f"timing.{k} missing")
    if not r["attempts"] or r["attempts"][-1]["level"] != r["level_used"]:
        p.append("the last attempt must be the returned picture")
    for a in r["attempts"]:
        if set(a) != {"level", "resolution", "seed", "time_s", "passed", "reasons"} or not isinstance(a["reasons"], list):
            p.append(f"attempt fields wrong: {sorted(a)}")
            break
    c = r["checks"]
    if not (c.get("clip_score") is None or isinstance(c["clip_score"], (int, float))):
        p.append("checks.clip_score type")
    if not (isinstance(c.get("visual", {}).get("ok"), bool) and isinstance(c["visual"].get("flags"), list)):
        p.append("checks.visual shape")
    for s in c.get("semantic", []):
        if s.get("kind") not in KINDS or not isinstance(s.get("label"), str) or not isinstance(s.get("ok"), bool):
            p.append(f"checks.semantic item wrong: {s}")
            break
    if not isinstance(r["router"].get("start_level"), int) or not (r["router"].get("confidence") is None or isinstance(r["router"]["confidence"], float)):
        p.append("router shape")
    if not (isinstance(r["polish"].get("applied"), bool) and isinstance(r["polish"].get("steps"), list)):
        p.append("polish shape")
    return p

def generate(prompt, mode="balanced", seed=None, timeout=180):
    body = {"prompt": prompt, "mode": mode}
    if seed is not None:
        body["seed"] = seed
    t0 = time.perf_counter()
    r = httpx.post(f"{BASE}/api/generate", json=body, timeout=timeout)
    return r, time.perf_counter() - t0

def to_image(data_url):
    return Image.open(io.BytesIO(base64.b64decode(data_url.split(",", 1)[1])))

DEMO = ["two dogs and one cat sitting on a sofa", "a red car to the left of a blue bus",
        "a cozy cabin in a snowy forest at night, warm light in the windows"]
contract_rows, shown = [], []
for prompt in DEMO:
    for mode in ("fast", "balanced", "quality"):
        r, wall = generate(prompt, mode, seed=7)
        body = r.json()
        problems = contract_problems(body) if r.status_code == 200 else [f"HTTP {r.status_code}: {body}"]
        contract_rows.append({"prompt": prompt[:40], "mode": mode, "ok": not problems, "problems": "; ".join(problems),
                              "server_s": body.get("timing", {}).get("total_s"), "tries": len(body.get("attempts", [])),
                              "passed": body.get("passed")})
        if r.status_code == 200:
            shown.append((to_image(body["image"]), f"{mode}: {body['timing']['total_s']:.2f} s, {len(body['attempts'])} "
                           f"{'try' if len(body['attempts']) == 1 else 'tries'}, {'passed' if body['passed'] else 'best effort'}\n"
                           + ", ".join(("✓ " if s["ok"] else "✗ ") + s["label"] for s in body["checks"]["semantic"]), prompt))
CONTRACT = pd.DataFrame(contract_rows)
display(CONTRACT)
print("Contract test:", "ALL RESPONSES MATCH" if CONTRACT.ok.all() else "PROBLEMS FOUND (see 'problems')")

fig, axes = plt.subplots(len(DEMO), 3, figsize=(12, 4.3 * len(DEMO)))
for ax, (im, title, prompt) in zip(np.array(axes).ravel(), shown):
    ax.imshow(im)
    ax.set_title(title, fontsize=7)
    ax.axis("off")
for i, prompt in enumerate(DEMO):
    np.array(axes).reshape(len(DEMO), 3)[i, 0].text(0, -60, prompt, fontsize=8, weight="bold")
plt.tight_layout()
plt.savefig(OUT / "demo_prompts.png", dpi=100)
plt.show()

## 7 · Error test: does the server answer every bad request the way the app expects?

In [ ]:
cases = [
    ("empty prompt", {"prompt": "   ", "mode": "fast"}, 400, "EMPTY_PROMPT"),
    ("prompt too long", {"prompt": "a " * 300, "mode": "fast"}, 400, "PROMPT_TOO_LONG"),
    ("unknown mode", {"prompt": "a cat", "mode": "turbo"}, 400, "BAD_MODE"),
    ("blocked prompt", {"prompt": "a nude person on a beach", "mode": "fast"}, 422, "UNSAFE_PROMPT"),
    ("not JSON", "hello", 400, "EMPTY_PROMPT"),
]
err_rows = []
for name, body, status, code in cases:
    if isinstance(body, dict):
        r = httpx.post(f"{BASE}/api/generate", json=body, timeout=30)
    else:
        r = httpx.post(f"{BASE}/api/generate", content=body, headers={"Content-Type": "application/json"}, timeout=30)
    got = r.json().get("error", {})
    err_rows.append({"case": name, "expected": f"{status} {code}", "got": f"{r.status_code} {got.get('code')}",
                     "ok": r.status_code == status and got.get("code") == code and bool(got.get("message"))})
r = httpx.get(f"{BASE}/api/nope", timeout=10)
err_rows.append({"case": "unknown endpoint", "expected": "404 JSON error", "got": f"{r.status_code} {r.json().get('error', {}).get('code')}",
                 "ok": r.status_code == 404 and "error" in r.json()})
ERRORS = pd.DataFrame(err_rows)
display(ERRORS)
print("Error test:", "ALL OK" if ERRORS.ok.all() else "PROBLEMS FOUND")

## 8 · Speed test: each mode on held-out prompts (server time, no phone network)
Compared with notebook 03a's measurement of the same modes (replayed from its picture-by-picture results).

In [ ]:
bench_rows = []
for mode in ("fast", "balanced", "quality"):
    for i, prompt in enumerate(BENCH):
        r, wall = generate(prompt, mode, seed=1000 + i)
        b = r.json()
        if r.status_code != 200:
            bench_rows.append({"mode": mode, "status": r.status_code})
            continue
        bench_rows.append({"mode": mode, "status": 200, "server_s": b["timing"]["total_s"], "wall_s": wall,
                           "tries": len(b["attempts"]), "passed": b["passed"], "level": b["level_used"],
                           "kb": len(b["image"]) * 3 / 4 / 1024})
B = pd.DataFrame(bench_rows)
ok = B[B.status == 200]
SPEED = ok.groupby("mode").agg(avg_s=("server_s", "mean"), p90_s=("server_s", lambda s: s.quantile(0.9)),
                               max_s=("server_s", "max"), avg_tries=("tries", "mean"), passed=("passed", "mean"),
                               avg_kb=("kb", "mean")).reindex(["fast", "balanced", "quality"])
for m in SPEED.index:
    measured = (CONFIG.get("modes", {}).get(m) or {}).get("measured") or {}
    SPEED.loc[m, "03a_replay_avg_s"] = measured.get("avg_s", np.nan)
    SPEED.loc[m, "03a_live_avg_s"] = measured.get("live_avg_s", np.nan)
display(SPEED.round(3))
print(f"Failed requests: {(B.status != 200).sum()} of {len(B)}")
B.to_csv(OUT / "speed_requests.csv", index=False)

## 9 · Several judges at once
The GPU draws one picture at a time; the others wait in a queue (up to 6, then the app says "busy, try again").

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(BURST) as ex:
    burst = list(ex.map(lambda i: generate(BENCH[i % len(BENCH)], "balanced", seed=i), range(BURST)))
codes = pd.Series([r.status_code for r, _ in burst]).value_counts().to_dict()
waits = sorted(w for r, w in burst if r.status_code == 200)
BURST_RESULT = {"requests": BURST, "status_codes": codes, "fastest_s": round(waits[0], 2) if waits else None,
                "slowest_s": round(waits[-1], 2) if waits else None}
print(BURST_RESULT)
print(f"With {BURST} people pressing Generate at the same moment, the last one waited {BURST_RESULT['slowest_s']} s.")

## 10 · GPU memory of the running server

In [ ]:
GPU_MB = None
try:
    out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    GPU_MB = sum(int(line.split(",")[1]) for line in out.strip().splitlines() if line and int(line.split(",")[0]) == server.pid)
    total = subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True).stdout.strip()
    print(f"Server process uses {GPU_MB / 1024:.2f} GB of the GPU's {int(total) / 1024:.1f} GB "
          "(includes PyTorch's reserved cache, so it's higher than the 'allocated' numbers in 03a).")
except Exception as e:
    print("nvidia-smi not available:", e)

## 11 · Save the tested library versions and a summary

In [ ]:
import importlib.metadata as md

PKGS = ["torch", "torchvision", "diffusers", "transformers", "accelerate", "bitsandbytes", "sentencepiece", "protobuf",
        "opencv-python-headless", "numpy", "pillow", "fastapi", "uvicorn"]
pins = []
for pkg in PKGS:
    version = None
    for name in ([pkg, "opencv-python"] if pkg == "opencv-python-headless" else [pkg]):
        try:
            version = md.version(name).split("+")[0]      # "2.6.0+cu124" -> "2.6.0" (the PyPI wheel is the CUDA build)
            break
        except md.PackageNotFoundError:
            pass
    req = "uvicorn[standard]" if pkg == "uvicorn" else pkg
    pins.append(f"{req}=={version}" if version else req)
REQ_TESTED = OUT / "requirements_tested.txt"
REQ_TESTED.write_text("# Versions that passed notebook 03b\n" + "\n".join(pins) + "\n", encoding="utf-8")
print(REQ_TESTED.read_text())

lines = ["# Notebook 03b: backend test", "",
         f"Config: {CONFIG_FILE.name} (generator setup {SETUP}). Server ready after {STARTUP_S:.0f} s (download + warm-up).", "",
         f"## Contract test: {'all responses match' if CONTRACT.ok.all() else 'PROBLEMS'}", "", CONTRACT.to_markdown(index=False), "",
         f"## Error test: {'all OK' if ERRORS.ok.all() else 'PROBLEMS'}", "", ERRORS.to_markdown(index=False), "",
         "## Speed per mode (server time, no network)", "", SPEED.round(3).to_markdown(), "",
         f"## Several judges at once: {BURST_RESULT}", "",
         f"GPU memory of the server process: {GPU_MB / 1024:.2f} GB" if GPU_MB else "GPU memory: n/a", "",
         "## Tested versions", "", "```", *pins, "```"]
(OUT / "03b_summary.md").write_text("\n".join(lines), encoding="utf-8")
print("Saved summary, versions and logs to", OUT)

## 12 · Deploy to your Hugging Face Space (only when `DEPLOY = True`)
**One-time set-up (you, in the browser):**
1. On huggingface.co create an **access token** with **write** permission (Settings → Access Tokens).
2. In Colab's 🔑 **Secrets** panel, add it as `HF_TOKEN` and switch on notebook access. Never paste it into a cell.
3. Set `HF_SPACE` in section 1 (e.g. `your-username/p3-studio`) and `DEPLOY = True`, then run this cell.

This uploads the server, the modules, the config, the pinned versions and the frontend **source** (the Space builds the phone app itself). It does **not** change the Space's hardware: you pick **T4 small** in the Space's settings, because that's what costs money.

In [ ]:
if DEPLOY:
    assert HF_SPACE and "/" in HF_SPACE, "Set HF_SPACE in section 1, e.g. 'your-username/p3-studio'"
    assert CONTRACT.ok.all() and ERRORS.ok.all(), "Tests failed above: fix them before deploying"
    from google.colab import userdata
    from huggingface_hub import HfApi
    sys.path.insert(0, str(SPACE))
    import assemble
    build = assemble.assemble(SPACE / CONFIG_FILE.name, Path("/content/space_build"), REQ_TESTED)
    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.create_repo(HF_SPACE, repo_type="space", space_sdk="docker", exist_ok=True)
    api.upload_folder(folder_path=str(build), repo_id=HF_SPACE, repo_type="space",
                      commit_message=f"Deploy from notebook 03b ({CONFIG_FILE.name}, {SETUP})")
    sub = HF_SPACE.replace("/", "-").replace("_", "-").replace(".", "-").lower()
    print(f"Uploaded. Build progress: https://huggingface.co/spaces/{HF_SPACE}")
    print(f"App link (once built and on a T4): https://{sub}.hf.space")
    print("Next: Space → Settings → Hardware → T4 small; set 'Sleep time' so it doesn't nap during judging.")
else:
    print("DEPLOY = False: nothing uploaded. Tests and versions are saved; set DEPLOY = True when ready.")

## 13 · Stop the server

In [ ]:
server.terminate()
try:
    server.wait(timeout=30)
except subprocess.TimeoutExpired:
    server.kill()
print("Server stopped.")